In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, roc_auc_score
import numpy as np
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

## Load and Prepare Data

In [ ]:
spotify_high_pop = pd.read_csv("/Users/donyabehroozi/Documents/gsb545/GSB-545/Project Files/high_popularity_spotify_data.csv")
spotify_high_pop.head()

In [ ]:
spotify_low_pop = pd.read_csv('/Users/donyabehroozi/Documents/gsb545/GSB-545/Project Files/low_popularity_spotify_data.csv')
spotify_low_pop.head()

In [ ]:
#make binary popular variable (1 = popular track, 0 = not a popular track)
spotify_high_pop["popular"] = 1
spotify_low_pop["popular"] = 0

#concatenate high and low popularity datasets
spotify_full = pd.concat([spotify_high_pop, spotify_low_pop], axis=0).reset_index(drop=True)

In [ ]:
#drop NAs
spotify_full = spotify_full.dropna()

#fix release year
def parse_release_year(date_str):
    if pd.isna(date_str):
        return None
    date_str = str(date_str).strip()
    try:
        return pd.to_datetime(date_str).year
    except:
        return None

spotify_full['release_year'] = spotify_full['track_album_release_date'].apply(parse_release_year)

#drop irrelevant columns
drop_cols = ['track_artist', 'track_href', 'uri', 'track_album_name',
             'playlist_name', 'analysis_url', 'track_id', 'track_name',
             'track_album_id', 'id', 'type', 'playlist_id',
             'track_album_release_date',
             'track_popularity']

spotify_full = spotify_full.drop(columns=drop_cols)

In [ ]:
#collapse playlist genres and subgenres
genre_map = {
    'electronic': 'electronic', 'ambient': 'electronic', 'lofi': 'electronic',
    'gaming': 'electronic', 'disco': 'electronic',
    'pop': 'pop', 'indie': 'pop', 'k-pop': 'pop', 'j-pop': 'pop',
    'cantopop': 'pop', 'mandopop': 'pop',
    'hip-hop': 'hip-hop_rnb', 'r&b': 'hip-hop_rnb', 'soul': 'hip-hop_rnb',
    'funk': 'hip-hop_rnb',
    'latin': 'latin_world', 'world': 'latin_world', 'arabic': 'latin_world',
    'brazilian': 'latin_world', 'afrobeats': 'latin_world', 'turkish': 'latin_world',
    'indian': 'latin_world', 'korean': 'latin_world', 'soca': 'latin_world',
    'reggae': 'latin_world',
    'rock': 'rock_metal', 'metal': 'rock_metal', 'punk': 'rock_metal',
    'jazz': 'jazz_blues', 'blues': 'jazz_blues', 'gospel': 'jazz_blues',
    'classical': 'classical_folk', 'folk': 'classical_folk',
    'wellness': 'wellness', 'country': 'wellness'
}

subgenre_map = {
    'chill': 'chill', 'lofi': 'chill', 'meditative': 'chill',
    'yoga': 'chill', 'soft': 'chill', 'bedroom': 'chill', 'smooth': 'chill',
    'modern': 'modern', 'mainstream': 'modern', 'pop': 'modern',
    'feel-good': 'modern', 'essential': 'modern',
    'classic': 'classic', 'throwback': 'classic', '80s': 'classic',
    '90s': 'classic', 'retro': 'classic',
    'hip-hop': 'urban', 'trap': 'urban', 'gangster': 'urban',
    'drill': 'urban', 'grime': 'urban',
    'deep house': 'electronic', 'techno': 'electronic', 'hardstyle': 'electronic',
    'future': 'electronic', 'vaporwave': 'electronic', 'future bass': 'electronic',
    'afro house': 'electronic',
    'reggaeton': 'latin', 'tropical': 'latin', 'afro-latin': 'latin',
    'cumbia': 'latin', 'samba': 'latin', 'forró': 'latin', 'carnival': 'latin',
    'french': 'world', 'scandi': 'world', 'nordic': 'world',
    'african': 'world', 'global': 'world', 'chinese': 'world',
    'japanese': 'world', 'nigerian': 'world', 'desi': 'world',
    'bhangra': 'world', 'bollywood': 'world', 'amapiano': 'world',
    'gqom': 'world', 'throat singing': 'world', 'australian': 'world',
    'celtic': 'world', 'irish': 'world', 'indigenous': 'world',
    'klezmer': 'world', 'jewish': 'world', 'tango': 'world',
    'cajun': 'world', 'southern': 'world', 'american': 'world',
    'classical': 'classical', 'neo-classical': 'classical', 'choral': 'classical',
    'cinematic': 'classical', 'academic': 'classical', 'soundtracks': 'classical',
    'noir': 'classical', 'drama': 'classical',
    'alternative': 'alternative', 'indie': 'alternative', 'pop punk': 'alternative',
    'death': 'alternative', 'heavy': 'alternative', 'experimental': 'alternative',
    'post-rock': 'alternative', 'avant-garde': 'alternative',
    'funk': 'other', 'melodic': 'other', 'workout': 'other',
    'anime': 'other', 'italo': 'other', 'delta': 'other',
    'fusion': 'other', 'latin': 'other', 'spanish': 'other',
    'afrobeats': 'other'
}

spotify_full['playlist_genre'] = spotify_full['playlist_genre'].map(genre_map)
spotify_full['playlist_subgenre'] = spotify_full['playlist_subgenre'].map(subgenre_map)

## Data Preprocessing

In [ ]:
quant_vars = [
    'energy', 'tempo', 'danceability', 'loudness',
    'liveness', 'valence', 'time_signature',
    'speechiness', 'instrumentalness',
    'mode', 'key', 'duration_ms', 'acousticness', 'release_year'
]

cat_vars = ['playlist_genre', 'playlist_subgenre']

y = spotify_full['popular']

X = pd.concat([
    spotify_full[quant_vars],
    pd.get_dummies(spotify_full[cat_vars], drop_first=True).astype(int)
], axis=1)

#train + test + validation split
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=321)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, stratify=y_train_full, test_size=0.2, random_state=321)

#check class balance before SMOTE
print("Class distribution before SMOTE:")
print(y_train.value_counts(normalize=True))

## SMOTE

SMOTE (Synthetic Minority Oversampling Technique) is applied only to the training set to address class imbalance. The validation and test sets are left untouched to preserve realistic evaluation conditions.

In [ ]:
#apply SMOTE to training data only
sm = SMOTE(random_state=321)
X_train_smote, y_train_smote = sm.fit_resample(X_train, y_train)

#check class balance after SMOTE
print("Class distribution after SMOTE:")
print(pd.Series(y_train_smote).value_counts(normalize=True))
print(f"\nTraining set size before SMOTE: {len(X_train)}")
print(f"Training set size after SMOTE:  {len(X_train_smote)}")

## SMOTE Baseline Models

Random Forest and LightGBM are evaluated using SMOTE-resampled training data in place of class weights. Cross-validation is conducted using an `imblearn` pipeline to ensure SMOTE is applied within each fold, preventing data leakage.

In [ ]:
#define cross validation method
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=321)

In [ ]:
#evaluate models - Random Forest with SMOTE (Validation data)

#use imblearn pipeline to apply SMOTE inside each CV fold
rf_smote_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=321)),
    ('model', RandomForestClassifier(random_state=321, n_jobs=-1))
])

cv_scores_rf_smote = cross_val_score(rf_smote_pipeline, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
print(cv_scores_rf_smote)
print(cv_scores_rf_smote.mean())
print(cv_scores_rf_smote.std())

#fit on full SMOTE training set and evaluate
rf_smote_model = RandomForestClassifier(random_state=321, n_jobs=-1)
rf_smote_model.fit(X_train_smote, y_train_smote)

y_proba = rf_smote_model.predict_proba(X_val)[:, 1]
print("Random Forest SMOTE Baseline ROC AUC (Validation):", roc_auc_score(y_val, y_proba))
print(classification_report(y_val, rf_smote_model.predict(X_val)))

In [ ]:
#evaluate models - Random Forest with SMOTE (Test data)
y_proba = rf_smote_model.predict_proba(X_test)[:, 1]
print("Random Forest SMOTE Baseline ROC AUC (Test):", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, rf_smote_model.predict(X_test)))

In [ ]:
#evaluate models - LightGBM with SMOTE (Validation data)

#use imblearn pipeline to apply SMOTE inside each CV fold
lgbm_smote_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=321)),
    ('model', LGBMClassifier(random_state=321, n_jobs=-1, verbose=-1))
])

cv_scores_lgbm_smote = cross_val_score(lgbm_smote_pipeline, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
print(cv_scores_lgbm_smote)
print(cv_scores_lgbm_smote.mean())
print(cv_scores_lgbm_smote.std())

#fit on full SMOTE training set and evaluate
lgbm_smote_model = LGBMClassifier(random_state=321, n_jobs=-1, verbose=-1)
lgbm_smote_model.fit(X_train_smote, y_train_smote)

y_proba = lgbm_smote_model.predict_proba(X_val)[:, 1]
print("LightGBM SMOTE Baseline ROC AUC (Validation):", roc_auc_score(y_val, y_proba))
print(classification_report(y_val, lgbm_smote_model.predict(X_val)))

In [ ]:
#evaluate models - LightGBM with SMOTE (Test data)
y_proba = lgbm_smote_model.predict_proba(X_test)[:, 1]
print("LightGBM SMOTE Baseline ROC AUC (Test):", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, lgbm_smote_model.predict(X_test)))

## SMOTE vs. Class Weights Comparison

Direct comparison of SMOTE baseline models against class weight baseline models from the original pipeline.

In [ ]:
#summary comparison table
results = {
    'Model': [
        'Random Forest - Class Weights (Baseline)',
        'Random Forest - SMOTE (Baseline)',
        'LightGBM - Class Weights (Baseline)',
        'LightGBM - SMOTE (Baseline)'
    ],
    'Validation ROC AUC': [
        0.0,  # fill in from original notebook
        roc_auc_score(y_val, rf_smote_model.predict_proba(X_val)[:, 1]),
        0.0,  # fill in from original notebook
        roc_auc_score(y_val, lgbm_smote_model.predict_proba(X_val)[:, 1])
    ],
    'Test ROC AUC': [
        0.922,  # from original notebook
        roc_auc_score(y_test, rf_smote_model.predict_proba(X_test)[:, 1]),
        0.927,  # from original notebook
        roc_auc_score(y_test, lgbm_smote_model.predict_proba(X_test)[:, 1])
    ]
}

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))